# Patching Sentinel-2 scenes for ML inference

This notebook is the **patcher slice** of the canonical cross-repo Lake
Tahoe scenario:

> **Cloud-free Sentinel-2 NDVI over Lake Tahoe, summer 2024.**

The full end-to-end story — catalog discovery via
[`geocatalog`](https://github.com/jejjohnson/geocatalog), operator
composition via [`geotoolz`](https://github.com/jejjohnson/geotoolz),
and patcher-based reconstruction via `geopatcher` — lives in
[`geocatalog/docs/notebooks/end_to_end_lake_tahoe.ipynb`](https://github.com/jejjohnson/geocatalog/blob/main/docs/notebooks/end_to_end_lake_tahoe.ipynb).

This notebook isolates the patcher: take **one Sentinel-2 scene**, split
it into 256×256 patches with 32-pixel overlap, apply a per-patch
operator, and stitch with `SpatialOverlapAdd`.

## Scenario parameters

| Parameter | Value |
|---|---|
| AOI bbox (EPSG:4326) | `(-120.25, 38.85, -119.85, 39.30)` |
| Date range | `2024-06-01` / `2024-09-30` |
| STAC root | `https://planetarycomputer.microsoft.com/api/stac/v1` |
| Collection | `sentinel-2-l2a` |
| Cloud cover | `< 20 %` |

> **Run locally.** This notebook is shipped un-executed. It hits the
> Microsoft Planetary Computer STAC API, which is unreachable from CI
> sandboxes. Open it locally with
> `pip install 'geopatcher[xarray-raster,streaming]' pystac-client planetary-computer rioxarray`
> and re-run all cells to materialise outputs.

In [ ]:
import dataclasses
from itertools import islice

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import planetary_computer
import pystac_client
import rioxarray
from georeader.rio_xarray_reader import RioXarrayReader

import geopatcher as gp

## 1. Load one Sentinel-2 scene

Search Microsoft Planetary Computer for the cloud-freest Sentinel-2
L2A scene over Lake Tahoe in summer 2024, then open the red band as a
lazy `rioxarray.DataArray` and wrap it as a `RasterField`.

In [ ]:
STAC = "https://planetarycomputer.microsoft.com/api/stac/v1"
BBOX = (-120.25, 38.85, -119.85, 39.30)

catalog = pystac_client.Client.open(STAC, modifier=planetary_computer.sign_inplace)
search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=BBOX,
    datetime="2024-06-01/2024-09-30",
    query={"eo:cloud_cover": {"lt": 20}},
)
items = sorted(search.items(), key=lambda it: it.properties["eo:cloud_cover"])
item = items[0]
print(f"scene: {item.id}")
print(f"cloud cover: {item.properties['eo:cloud_cover']:.2f}%")
print(f"red asset: {item.assets['B04'].href[:80]}…")

In [ ]:
red = rioxarray.open_rasterio(
    item.assets["B04"].href,
    masked=True,
    chunks={"x": 1024, "y": 1024},
)
print(f"red.shape: {red.shape}")
print(f"red.rio.crs: {red.rio.crs}")
print(f"red.rio.resolution(): {red.rio.resolution()}")

reader = RioXarrayReader(red)
field = gp.RasterField(reader)

## 2. Define a `SpatialPatcher`

256×256 patches with **32-pixel overlap** (so the sampler step is
`256 - 32 = 224`), a `SpatialHann` window to feather the seams, and
`SpatialOverlapAdd` for the stitch.

(`SpatialHann` is the geopatcher analogue of the "feather" weight common
in tiled inference — it tapers smoothly to zero at the patch boundary so
OverlapAdd's denominator hides the seams.)

In [ ]:
PATCH_SIZE = 256
OVERLAP = 32
STEP = PATCH_SIZE - OVERLAP

patcher = gp.SpatialPatcher(
    geometry=gp.SpatialRectangular(size=(PATCH_SIZE, PATCH_SIZE)),
    sampler=gp.SpatialRegularStride(step=(STEP, STEP)),
    window=gp.SpatialHann(),
    aggregation=gp.SpatialOverlapAdd(),
)
print(patcher.get_config())
print(f"n_anchors: {patcher.n_anchors(field)}")

## 3. `patcher.split(field)` — yields patches one at a time

Streaming is the default — `split` returns an `Iterator[Patch]`. We
peek at the first few patches and overlay the patch grid on the scene
so the anchor schedule is visible.

In [ ]:
first = list(islice(patcher.split(field), 3))
for p in first:
    arr = np.asarray(p.data)
    print(
        f"anchor={p.anchor}  indices={p.indices}  data.shape={arr.shape}  dtype={arr.dtype}"
    )

In [ ]:
# Read a coarse preview of the scene for context, then overlay the patch grid.
preview = np.asarray(field.domain)[0]
H, W = preview.shape

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(
    preview,
    cmap="Greys_r",
    vmin=np.nanpercentile(preview, 2),
    vmax=np.nanpercentile(preview, 98),
)
for anchor in patcher.anchors(field)[:120]:  # cap at 120 for readability
    r0, c0 = anchor
    ax.add_patch(
        mpatches.Rectangle(
            (c0, r0), PATCH_SIZE, PATCH_SIZE, fill=False, edgecolor="orange", lw=0.5
        )
    )
ax.set_title(f"Patch grid on Sentinel-2 B04 (first 120 of {patcher.n_anchors(field)})")
ax.set_xticks([])
ax.set_yticks([])
plt.show()

## 4. Per-patch operator — channel normalisation

A toy operator that L1-normalises each chip independently. Real
ML pipelines slot a trained model here; the patcher contract is the
same.

In [ ]:
def normalise(arr) -> np.ndarray:
    a = np.asarray(arr, dtype=np.float32)
    mu = np.nanmean(a)
    sigma = np.nanstd(a) + 1e-6
    return (a - mu) / sigma


outputs = []
for patch in patcher.split(field):
    new_data = normalise(patch.data)
    outputs.append(dataclasses.replace(patch, data=new_data))
print(f"materialised {len(outputs)} patches")

## 5. Stitch with `SpatialOverlapAdd`

The `SpatialHann` window weights the centre of each patch higher than
its edge; `SpatialOverlapAdd` accumulates the weighted contributions
and divides by the sum-of-weights, hiding the 32-px seams.

For >1 TB outputs the same call accumulates to a chunked zarr store on
disk — swap `SpatialOverlapAdd()` for
`SpatialOverlapAdd(streaming=True, target_path="out.zarr", chunks=(256, 256))`.
See [`recipes/streaming-overlap-add.md`](../recipes/streaming-overlap-add.md).

In [ ]:
stitched = patcher.merge(outputs, field.domain)
print(f"stitched.shape: {stitched.shape}, dtype: {stitched.dtype}")

## 6. Input vs output, side by side

Sanity-check the reconstruction: the per-patch normalisation should be
visually obvious (each chip's local distribution centred and scaled) but
the seams should be invisible thanks to the Hann taper.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(
    preview,
    cmap="viridis",
    vmin=np.nanpercentile(preview, 2),
    vmax=np.nanpercentile(preview, 98),
)
axes[0].set_title(f"Input — Sentinel-2 B04\n{preview.shape}")
stitched_arr = np.asarray(stitched)
axes[1].imshow(
    stitched_arr,
    cmap="viridis",
    vmin=np.nanpercentile(stitched_arr, 2),
    vmax=np.nanpercentile(stitched_arr, 98),
)
axes[1].set_title(f"Per-patch normalised\n{stitched_arr.shape}")
for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])
plt.show()

## 7. Next — the full end-to-end story

This notebook covered the **patcher** axis only: split → operate →
stitch on a single Sentinel-2 scene.

The canonical cross-repo notebook stitches all three packages together:

- [`geocatalog`](https://github.com/jejjohnson/geocatalog) discovers
  Sentinel-2 items via STAC, ranks them by cloud cover, and resolves
  asset URLs.
- [`geotoolz`](https://github.com/jejjohnson/geotoolz) composes the
  per-patch normalisation + NDVI calculation as a lazy `Sequential`
  pipeline.
- `geopatcher` drives the locality layer — exactly what this notebook
  demonstrates.

**Read it here:**
[`geocatalog/docs/notebooks/end_to_end_lake_tahoe.ipynb`](https://github.com/jejjohnson/geocatalog/blob/main/docs/notebooks/end_to_end_lake_tahoe.ipynb)

### Further reading

- [Concepts](../concepts.md) — the four-axis abstraction in full.
- [Recipes / streaming overlap-add](../recipes/streaming-overlap-add.md) — bounded-memory pipelines for >1 TB outputs.
- [Recipes / on-error policies](../recipes/on-error-policies.md) — raise / skip / mask / retry.
- [Recipes / journal & resume](../recipes/journal-and-resume.md) — `PatchJournal` for restartable jobs.